In [1]:
import torch
from src.cocoruta_sayless import print_env_config, get_tokenizer_and_model, query_model, say_less, build_prompt

%load_ext autoreload
%autoreload 2

print_env_config()

/home/marcos.moretti/repos/conformal-factual-lm/venv-cf-2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda available: True
cuda devices: 4
NVIDIA RTX 4000 Ada Generation
NVIDIA RTX 4000 Ada Generation
NVIDIA RTX 4000 Ada Generation
NVIDIA RTX 4000 Ada Generation
torch cuda version: 12.8
2.10.0+cu128


## Usando modelo do Cocoruta

In [2]:
model_id = "felipeoes/cocoruta-7b" #"meta-llama/Llama-3.1-8B"

In [ ]:
# using the second GPU only
tokenizer, model = get_tokenizer_and_model(
    model_id=model_id, device_map="cuda:1", torch_dtype=torch.float16)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 74.11it/s, Materializing param=model.norm.weight]                              


## Rodando com parâmetros recomendados no Git do Cocoruta

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, StoppingCriteria

# Define an early stopping ccriteria
class StopOnString(StoppingCriteria):
    def __init__(self, target_sequence, prompt):
        self.target_sequence = target_sequence
        self.prompt=prompt

    def __call__(self, input_ids, scores, **kwargs):
        # Get the generated text as a string
        generated_text = tokenizer.decode(input_ids[0])
        generated_text = generated_text.replace(self.prompt,'')
        # Check if the target sequence appears in the generated text
        if self.target_sequence in generated_text:
            return True  # Stop generation

        return False  # Continue generation

# INFERENCE
stop_string = "###"
streamer = TextStreamer(tokenizer, skip_prompt=True)

input_text = "### Pergunta: O que é a Amazônia Azul?\n### Resposta:"
input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(model.device)
_ = model.generate(input_ids,
                            streamer=streamer,
                            pad_token_id=tokenizer.eos_token_id, 
                            max_length=256,
                            temperature=0.8,
                            top_p=0.9,
                            top_k=30,
                            stopping_criteria=[StopOnString(stop_string, input_text)],
                            do_sample=True,
                            num_return_sequences=1,
)

# Remove the prompt part
generated_ids = _[:, input_ids.shape[-1]:]

# Decode only the generated tokens
output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(output_text)

tokenizer.decode(_[0], skip_special_tokens=True)


A Amazônia Azul é a região marítima brasileira que compreende a Zona Econômica Exclusiva (ZEE) e a Plataforma Continental. A ZEE é a área do mar adjacente ao território do Brasil, com largura de até 200 milhas marítimas, e a Plataforma Continental é a extensão do território brasileiro sob o mar, até o limite de 200 metros de profundidade ou até o limite de 350 milhas marítimas, se a profundidade for menor que 200 metros.

###

A Amazônia Azul é a região marítima brasileira que compreende a Zona Econômica Exclusiva (ZEE) e a Plataforma Continental. A ZEE é a área do mar adjacente ao território do Brasil, com largura de até 200 milhas marítimas, e a Plataforma Continental é a extensão do território brasileiro sob o mar, até o limite de 200 metros de profundidade ou até o limite de 350 milhas marítimas, se a profundidade for menor que 200 metros.

###


'### Pergunta: O que é a Amazônia Azul?\n### Resposta:\nA Amazônia Azul é a região marítima brasileira que compreende a Zona Econômica Exclusiva (ZEE) e a Plataforma Continental. A ZEE é a área do mar adjacente ao território do Brasil, com largura de até 200 milhas marítimas, e a Plataforma Continental é a extensão do território brasileiro sob o mar, até o limite de 200 metros de profundidade ou até o limite de 350 milhas marítimas, se a profundidade for menor que 200 metros.\n\n###'

6,5 segundos

## Testando com uma pergunta maior

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, StoppingCriteria

# Define an early stopping ccriteria
class StopOnString(StoppingCriteria):
    def __init__(self, target_sequence, prompt):
        self.target_sequence = target_sequence
        self.prompt=prompt

    def __call__(self, input_ids, scores, **kwargs):
        # Get the generated text as a string
        generated_text = tokenizer.decode(input_ids[0])
        generated_text = generated_text.replace(self.prompt,'')
        # Check if the target sequence appears in the generated text
        if self.target_sequence in generated_text:
            return True  # Stop generation

        return False  # Continue generation

# INFERENCE
stop_string = "###"
streamer = TextStreamer(tokenizer, skip_prompt=True)

input_text = "### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?\n### Resposta:"
input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(model.device)
_ = model.generate(input_ids,
                            streamer=streamer,
                            pad_token_id=tokenizer.eos_token_id, 
                            max_length=256,
                            temperature=0.8,
                            top_p=0.9,
                            top_k=30,
                            stopping_criteria=[StopOnString(stop_string, input_text)],
                            do_sample=True,
                            num_return_sequences=1,
)


# Remove the prompt part
generated_ids = _[:, input_ids.shape[-1]:]

# Decode only the generated tokens
output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(output_text)

tokenizer.decode(_[0], skip_special_tokens=True)


A Lei nº 12.712/2012 foi um marco na história da Amazônia Azul, pois promoveu a conscientização da população brasileira sobre a importância da conservação dos recursos naturais da região. A lei também contribuiu para a implementação de novas ações de proteção dos recifes de corais, como a proibição da pesca de explosivos e a promoção de projetos de reflorestamento de áreas costeiras.

###

A Lei nº 12.712/2012 foi um marco na história da Amazônia Azul, pois promoveu a conscientização da população brasileira sobre a importância da conservação dos recursos naturais da região. A lei também contribuiu para a implementação de novas ações de proteção dos recifes de corais, como a proibição da pesca de explosivos e a promoção de projetos de reflorestamento de áreas costeiras.

###


'### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?\n### Resposta:\nA Lei nº 12.712/2012 foi um marco na história da Amazônia Azul, pois promoveu a conscientização da população brasileira sobre a importância da conservação dos recursos naturais da região. A lei também contribuiu para a implementação de novas ações de proteção dos recifes de corais, como a proibição da pesca de explosivos e a promoção de projetos de reflorestamento de áreas costeiras.\n\n###'

5,5 segundos

## Mesma pergunta, mas com temperatura baixa

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, StoppingCriteria

# Define an early stopping ccriteria
class StopOnString(StoppingCriteria):
    def __init__(self, target_sequence, prompt):
        self.target_sequence = target_sequence
        self.prompt=prompt

    def __call__(self, input_ids, scores, **kwargs):
        # Get the generated text as a string
        generated_text = tokenizer.decode(input_ids[0])
        generated_text = generated_text.replace(self.prompt,'')
        # Check if the target sequence appears in the generated text
        if self.target_sequence in generated_text:
            return True  # Stop generation

        return False  # Continue generation

# INFERENCE
stop_string = "###"
streamer = TextStreamer(tokenizer, skip_prompt=True)

input_text = "### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?\n### Resposta:"
input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(model.device)
_ = model.generate(input_ids,
                            #streamer=streamer,
                            pad_token_id=tokenizer.eos_token_id, 
                            max_length=256,
                            temperature=1e-8,
                            top_p=0.9,
                            top_k=30,
                            stopping_criteria=[StopOnString(stop_string, input_text)],
                            do_sample=True,
                            num_return_sequences=1,
)

# Remove the prompt part
generated_ids = _[:, input_ids.shape[-1]:]

# Decode only the generated tokens
output_text1 = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(output_text1)


A lei não prevê nenhuma penalidade para infrações sobre a preservação dos recifes de corais na Amazônia Azul. No entanto, é possível que as infrações relacionadas à preservação dos recifes de corais sejam consideradas em virtude de outras leis, como a de Proteção à Flora e Fauna ou a de Controle da Poluição.

###


In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, StoppingCriteria

# Define an early stopping ccriteria
class StopOnString(StoppingCriteria):
    def __init__(self, target_sequence, prompt):
        self.target_sequence = target_sequence
        self.prompt=prompt

    def __call__(self, input_ids, scores, **kwargs):
        # Get the generated text as a string
        generated_text = tokenizer.decode(input_ids[0])
        generated_text = generated_text.replace(self.prompt,'')
        # Check if the target sequence appears in the generated text
        if self.target_sequence in generated_text:
            return True  # Stop generation

        return False  # Continue generation

# INFERENCE
stop_string = "###"
streamer = TextStreamer(tokenizer, skip_prompt=True)

input_text = "### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?\n### Resposta:"
input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(model.device)
_ = model.generate(input_ids,
                            #streamer=streamer,
                            pad_token_id=tokenizer.eos_token_id, 
                            max_length=256,
                            temperature=1e-8,
                            top_p=0.9,
                            top_k=30,
                            stopping_criteria=[StopOnString(stop_string, input_text)],
                            do_sample=True,
                            num_return_sequences=1,
)

# Remove the prompt part
generated_ids = _[:, input_ids.shape[-1]:]

# Decode only the generated tokens
output_text2 = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(output_text2)


A Lei nº 12.721/2012 tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul a seguinte disposição:

* **O artigo 11, § 3º, da Lei nº 12.721/2012 determina que as atividades de turismo recreativo na zona de vida silvestre das secções de arquipélagos dos Cocos e Rolas, no perímetro superior do limite de variação admissível, só serão permitidas com prioridade para beneficiários com vistas a empreendimentos de sustentabilidade econômica e com gestão ambientalmente consciente.**

Essa disposição visa a proteger os recifes de corais da Amazônia Azul, que são um


In [16]:
output_text1 == output_text2

False

Temperatura baixa, com do_sample=True, não garante replicabilidade

## Remove o streamer e troca `max_length=256` por `max_new_tokens=1000`

In [18]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, StoppingCriteria

# Define an early stopping ccriteria
class StopOnString(StoppingCriteria):
    def __init__(self, target_sequence, prompt):
        self.target_sequence = target_sequence
        self.prompt=prompt

    def __call__(self, input_ids, scores, **kwargs):
        # Get the generated text as a string
        generated_text = tokenizer.decode(input_ids[0])
        generated_text = generated_text.replace(self.prompt,'')
        # Check if the target sequence appears in the generated text
        if self.target_sequence in generated_text:
            return True  # Stop generation

        return False  # Continue generation

# INFERENCE
stop_string = "###"
streamer = TextStreamer(tokenizer, skip_prompt=True)

input_text = "### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?\n### Resposta:"
input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(model.device)
_ = model.generate(input_ids,
                            #streamer=streamer,
                            pad_token_id=tokenizer.eos_token_id, 
                            #max_length=256,
                            max_new_tokens=1000,
                            temperature=1e-8,
                            top_p=0.9,
                            top_k=30,
                            stopping_criteria=[StopOnString(stop_string, input_text)],
                            do_sample=True,
                            num_return_sequences=1,
)

# Remove the prompt part
generated_ids = _[:, input_ids.shape[-1]:]

# Decode only the generated tokens
output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(output_text)


A Lei nº 12.721/2012 tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul a seguinte disposição:

* **O artigo 11, § 3º, da Lei nº 12.721/2012 determina que as atividades de turismo recreativo na zona de vida silvestre das secções de arquipélagos dos Cocos e Rolas, no perímetro superior do limite de variação admissível, só serão permitidas com prioridade para beneficiários com vistas a empreendimentos de sustentabilidade econômica e com gestão ambientalmente consciente.**

Essa disposição visa a proteger os recifes de corais da Amazônia Azul, que são um dos mais importantes do mundo.

###


Não truncou a resposta!

## Usando `do_sample=False`

In [19]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, StoppingCriteria

# Define an early stopping ccriteria
class StopOnString(StoppingCriteria):
    def __init__(self, target_sequence, prompt):
        self.target_sequence = target_sequence
        self.prompt=prompt

    def __call__(self, input_ids, scores, **kwargs):
        # Get the generated text as a string
        generated_text = tokenizer.decode(input_ids[0])
        generated_text = generated_text.replace(self.prompt,'')
        # Check if the target sequence appears in the generated text
        if self.target_sequence in generated_text:
            return True  # Stop generation

        return False  # Continue generation

# INFERENCE
stop_string = "###"
streamer = TextStreamer(tokenizer, skip_prompt=True)

input_text = "### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?\n### Resposta:"
input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(model.device)
_ = model.generate(
            input_ids,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=1000,
            num_return_sequences=1,
            stopping_criteria=[StopOnString(stop_string, input_text)],
            do_sample=False,
)

# Remove the prompt part
generated_ids = _[:, input_ids.shape[-1]:]

# Decode only the generated tokens
output_text1 = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(output_text1)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



A Lei nº 12.721/2012 tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul a seguinte disposição:

* **O artigo 11, § 3º, da Lei nº 12.721/2012 determina que as atividades de turismo recreativo na zona de vida silvestre das secções de arquipélagos dos Cocos e Rolas, no perímetro superior do limite de variação admissível, só serão permitidas com prioridade para beneficiários com vistas a empreendimentos de sustentabilidade econômica e com gestão ambientalmente consciente.**

Essa disposição visa a proteger os recifes de corais da Amazônia Azul, que são um dos mais importantes do mundo.

###


In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, StoppingCriteria

# Define an early stopping ccriteria
class StopOnString(StoppingCriteria):
    def __init__(self, target_sequence, prompt):
        self.target_sequence = target_sequence
        self.prompt=prompt

    def __call__(self, input_ids, scores, **kwargs):
        # Get the generated text as a string
        generated_text = tokenizer.decode(input_ids[0])
        generated_text = generated_text.replace(self.prompt,'')
        # Check if the target sequence appears in the generated text
        if self.target_sequence in generated_text:
            return True  # Stop generation

        return False  # Continue generation

# INFERENCE
stop_string = "###"
streamer = TextStreamer(tokenizer, skip_prompt=True)

input_text = "### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?\n### Resposta:"
input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(model.device)
_ = model.generate(
            input_ids,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=1000,
            num_return_sequences=1,
            stopping_criteria=[StopOnString(stop_string, input_text)],
            do_sample=False,
)

# Remove the prompt part
generated_ids = _[:, input_ids.shape[-1]:]

# Decode only the generated tokens
output_text2 = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(output_text2)


A Lei nº 12.721/2012 tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul a seguinte disposição:

* **O artigo 11, § 3º, da Lei nº 12.721/2012 determina que as atividades de turismo recreativo na zona de vida silvestre das secções de arquipélagos dos Cocos e Rolas, no perímetro superior do limite de variação admissível, só serão permitidas com prioridade para beneficiários com vistas a empreendimentos de sustentabilidade econômica e com gestão ambientalmente consciente.**

Essa disposição visa a proteger os recifes de corais da Amazônia Azul, que são um dos mais importantes do mundo.

###


In [21]:
output_text1 == output_text2

True

In [22]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, StoppingCriteria

# Define an early stopping ccriteria
class StopOnString(StoppingCriteria):
    def __init__(self, target_sequence, prompt):
        self.target_sequence = target_sequence
        self.prompt=prompt

    def __call__(self, input_ids, scores, **kwargs):
        # Get the generated text as a string
        generated_text = tokenizer.decode(input_ids[0])
        generated_text = generated_text.replace(self.prompt,'')
        # Check if the target sequence appears in the generated text
        if self.target_sequence in generated_text:
            return True  # Stop generation

        return False  # Continue generation

# INFERENCE
stop_string = "###"
streamer = TextStreamer(tokenizer, skip_prompt=True)

input_text = "### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?\n### Resposta:"
input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(model.device)
_ = model.generate(
            input_ids,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=1000,
            num_return_sequences=1,
            stopping_criteria=[StopOnString(stop_string, input_text)],
            do_sample=False,
)

# Remove the prompt part
generated_ids = _[:, input_ids.shape[-1]:]

# Decode only the generated tokens
output_text3 = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(output_text3)


A Lei nº 12.721/2012 tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul a seguinte disposição:

* **O artigo 11, § 3º, da Lei nº 12.721/2012 determina que as atividades de turismo recreativo na zona de vida silvestre das secções de arquipélagos dos Cocos e Rolas, no perímetro superior do limite de variação admissível, só serão permitidas com prioridade para beneficiários com vistas a empreendimentos de sustentabilidade econômica e com gestão ambientalmente consciente.**

Essa disposição visa a proteger os recifes de corais da Amazônia Azul, que são um dos mais importantes do mundo.

###


In [23]:
output_text1 == output_text3

True

do_sample=False garante a replicabilidade

## Atualizei a função `query_model` do script

In [24]:
question1 = "O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?"
prompt1 = build_prompt(question1)
print(prompt1)
output_text1 = query_model(model, tokenizer, prompt1, max_tokens=1000, temperature=0, n_samples=1)
output_text2 = query_model(model, tokenizer, prompt1, max_tokens=1000, temperature=0, n_samples=1)
output_text3 = query_model(model, tokenizer, prompt1, max_tokens=1000, temperature=0, n_samples=1)
print(output_text1)
print(output_text2)
print(output_text3)
print(output_text1 == output_text2)
print(output_text1 == output_text3)

### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?
### Resposta:

A Lei nº 12.721/2012 tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul a seguinte disposição:

* **O artigo 11, § 3º, da Lei nº 12.721/2012 determina que as atividades de turismo recreativo na zona de vida silvestre das secções de arquipélagos dos Cocos e Rolas, no perímetro superior do limite de variação admissível, só serão permitidas com prioridade para beneficiários com vistas a empreendimentos de sustentabilidade econômica e com gestão ambientalmente consciente.**

Essa disposição visa a proteger os recifes de corais da Amazônia Azul, que são um dos mais importantes do mundo.

###

A Lei nº 12.721/2012 tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul a seguinte disposição:

* **O artigo 11, § 3º, da Lei nº 12.721/2012 determina que as a

In [25]:
question1 = "O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?"
prompt1 = build_prompt(question1)
print(prompt1)
output_text1 = query_model(model, tokenizer, prompt1, max_tokens=1000, temperature=0.8, n_samples=1)
output_text2 = query_model(model, tokenizer, prompt1, max_tokens=1000, temperature=0.8, n_samples=1)
output_text3 = query_model(model, tokenizer, prompt1, max_tokens=1000, temperature=0.8, n_samples=1)
print(output_text1)
print(output_text2)
print(output_text3)
print(output_text1 == output_text2)
print(output_text1 == output_text3)

### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?
### Resposta:

A lei não estabelece nenhuma penalidade para a pesca predatória de corais. No entanto, a lei estabelece que é de responsabilidade do pescador pescar coral em situações em que haja pesca predatória. Portanto, é possível que o pescador seja responsabilizado civilmente por danos causados ao meio ambiente, caso seja comprovada a pesca predatória.

###

A necessidade de preservar os recifes de corais na Amazônia Azul é refletida na legislação referente à região. O Decreto-Lei nº 2.210, de 28 de junho de 1940, que dispõe sobre a concessão de permissão para a pesca em embarcações estrangeiras, estabelece que é vedado o escoamento de resíduos sólidos ou líquidos de fábricas ou navios, bem como o lançamento de resíduos sólidos ou líquidos nos corpos dágua, inclusive em zonas de maré. O Decreto-Lei nº 5.194, de 15 de janeiro de 1943, que dispõe sobre a conce

In [26]:
question2 = "O que é a Amazônia Azul?"
prompt2 = build_prompt(question2)
print(prompt2)
output_text1 = query_model(model, tokenizer, prompt2, max_tokens=1000, temperature=0, n_samples=1)
output_text2 = query_model(model, tokenizer, prompt2, max_tokens=1000, temperature=0, n_samples=1)
output_text3 = query_model(model, tokenizer, prompt2, max_tokens=1000, temperature=0, n_samples=1)
print(output_text1)
print(output_text2)
print(output_text3)
print(output_text1 == output_text2)
print(output_text1 == output_text3)

### Pergunta: O que é a Amazônia Azul?
### Resposta:

A Amazônia Azul é a região marítima brasileira que compreende a Zona Econômica Exclusiva (ZEE) e a Plataforma Continental. A ZEE é a área do mar adjacente ao território do Brasil, com largura de até 200 milhas marítimas, e a Plataforma Continental é a extensão do território brasileiro sob o mar, até o limite de 200 metros de profundidade ou até o limite de 350 milhas marítimas, se a profundidade for menor que 200 metros.

###

A Amazônia Azul é a região marítima brasileira que compreende a Zona Econômica Exclusiva (ZEE) e a Plataforma Continental. A ZEE é a área do mar adjacente ao território do Brasil, com largura de até 200 milhas marítimas, e a Plataforma Continental é a extensão do território brasileiro sob o mar, até o limite de 200 metros de profundidade ou até o limite de 350 milhas marítimas, se a profundidade for menor que 200 metros.

###

A Amazônia Azul é a região marítima brasileira que compreende a Zona Econômica Exclu

In [29]:
question2 = "O que é a Amazônia Azul?"
prompt2 = build_prompt(question2)
print(prompt2)
output_text1 = query_model(model, tokenizer, prompt2, max_tokens=1000, temperature=0.8, n_samples=1)
output_text2 = query_model(model, tokenizer, prompt2, max_tokens=1000, temperature=0.8, n_samples=1)
output_text3 = query_model(model, tokenizer, prompt2, max_tokens=1000, temperature=0.8, n_samples=1)
print(output_text1)
print(output_text2)
print(output_text3)
print(output_text1 == output_text2)
print(output_text1 == output_text3)

### Pergunta: O que é a Amazônia Azul?
### Resposta:

A Amazônia Azul é a região marítima brasileira que compreende a Zona Econômica Exclusiva (ZEE) e a Plataforma Continental. A ZEE é a área do mar adjacente ao território do Brasil, com largura de até 200 milhas marítimas, e a Plataforma Continental é a extensão do território brasileiro sob o mar, até o limite de 200 metros de profundidade ou até o limite de 350 milhas marítimas, se a profundidade for menor que 200 metros.

###

A Amazônia Azul é a região marítima brasileira que compreende a Zona Econômica Exclusiva (ZEE) e a Plataforma Continental. A ZEE é a área do mar adjacente ao território do Brasil, com largura de até 200 milhas marítimas, e a Plataforma Continental é a extensão do território brasileiro sob o mar, até o limite de 200 metros de profundidade ou até o limite de 350 milhas marítimas, se a profundidade for menor que 200 metros.

###

A Amazônia Azul é a região marítima brasileira que compreende a Zona Econômica Exclu

In [30]:
# Para rodar todas as 16.000 perguntas
16000 * (9 + 6) / 2 / 60 / 60

33.333333333333336

## Tentando usar `device_map="auto"`

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, StoppingCriteria

n_gpus = torch.cuda.device_count()
# max_memory = f'{40960}MB'
# each gpu has 80 gb
max_memory = f'{76 * 1024}MB'

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",  # dispatch efficiently the model on the available resources
    # device_map={"": 0},
    #max_memory={i: max_memory for i in range(n_gpus)},
    torch_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    device_map="auto",
)

print(model.device)


Loading weights: 100%|██████████| 291/291 [00:01<00:00, 266.51it/s, Materializing param=model.norm.weight]                              


cuda:0


In [ ]:
question2 = "O que é a Amazônia Azul?"
prompt2 = build_prompt(question2)
print(prompt2)
class StopOnString(StoppingCriteria):
    def __init__(self, target_sequence, prompt):
        self.target_sequence = target_sequence
        self.prompt = prompt

    def __call__(self, input_ids, scores, **kwargs):
        # Get the generated text as a string
        generated_text = tokenizer.decode(input_ids[0])
        generated_text = generated_text.replace(self.prompt, "")
        # Check if the target sequence appears in the generated text
        if self.target_sequence in generated_text:
            return True  # Stop generation

        return False  # Continue generation

stop_string = "###"
input_ids = tokenizer(prompt2, return_tensors="pt").input_ids.to(model.device)

### Pergunta: O que é a Amazônia Azul?
### Resposta:


In [10]:
_ = model.generate(
    input_ids,
    pad_token_id=tokenizer.eos_token_id,
    max_new_tokens=1000,
    num_return_sequences=1,
    stopping_criteria=[StopOnString(stop_string, prompt2)],
    do_sample=False,
)
generated_ids = _[:, input_ids.shape[-1] :]

output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(output)

KeyboardInterrupt: 

Demora minutos e não finaliza...

## Tentando usar o quantizer para reduzir o tempo de rodagem

Obs.: isso reduz um pouco a qualidade

8-bit

99% das tarefas ficam praticamente iguais

Diferença mínima em:

geração criativa

raciocínio muito longo

Para uso normal → quase imperceptível

👉 Eu recomendaria 8-bit sem medo.

In [3]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, StoppingCriteria
tokenizer = AutoTokenizer.from_pretrained(model_id, device_map="cuda:1")
model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="cuda:1", quantization_config=bnb_config,
    #torch_dtype=torch.float16, (não precisa! os pesos serão carregados em INT8)
)

Loading weights: 100%|██████████| 291/291 [00:07<00:00, 40.70it/s, Materializing param=model.norm.weight]                              


In [6]:
question2 = "O que é a Amazônia Azul?"
prompt2 = build_prompt(question2)
print(prompt2)
class StopOnString(StoppingCriteria):
    def __init__(self, target_sequence, prompt):
        self.target_sequence = target_sequence
        self.prompt = prompt

    def __call__(self, input_ids, scores, **kwargs):
        # Get the generated text as a string
        generated_text = tokenizer.decode(input_ids[0])
        generated_text = generated_text.replace(self.prompt, "")
        # Check if the target sequence appears in the generated text
        if self.target_sequence in generated_text:
            return True  # Stop generation

        return False  # Continue generation

stop_string = "###"
input_ids = tokenizer(prompt2, return_tensors="pt").input_ids.to(model.device)

### Pergunta: O que é a Amazônia Azul?
### Resposta:


In [7]:
_ = model.generate(
    input_ids,
    pad_token_id=tokenizer.eos_token_id,
    max_new_tokens=1000,
    num_return_sequences=1,
    stopping_criteria=[StopOnString(stop_string, prompt2)],
    do_sample=False,
)
generated_ids = _[:, input_ids.shape[-1] :]

output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(output)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



A Amazônia Azul é a região marítima brasileira que compreende a Zona Econômica Exclusiva (ZEE) e a Plataforma Continental. A ZEE é a área do mar adjacente ao território do Brasil, com largura de até 200 milhas marítimas, e a Plataforma Continental é a extensão do território brasileiro sob o mar, até o limite de 200 metros de profundidade ou até o limite de 350 milhas marítimas, se a profundidade for menor que 200 metros.

###


In [8]:
question2 = "O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?"
prompt2 = build_prompt(question2)
print(prompt2)
class StopOnString(StoppingCriteria):
    def __init__(self, target_sequence, prompt):
        self.target_sequence = target_sequence
        self.prompt = prompt

    def __call__(self, input_ids, scores, **kwargs):
        # Get the generated text as a string
        generated_text = tokenizer.decode(input_ids[0])
        generated_text = generated_text.replace(self.prompt, "")
        # Check if the target sequence appears in the generated text
        if self.target_sequence in generated_text:
            return True  # Stop generation

        return False  # Continue generation

stop_string = "###"
input_ids = tokenizer(prompt2, return_tensors="pt").input_ids.to(model.device)

### Pergunta: O que tem de mais relevante sobre a preservação dos recifes de corais na legislação referente à Amazônia Azul?
### Resposta:


In [9]:
_ = model.generate(
    input_ids,
    pad_token_id=tokenizer.eos_token_id,
    max_new_tokens=1000,
    num_return_sequences=1,
    stopping_criteria=[StopOnString(stop_string, prompt2)],
    do_sample=False,
)
generated_ids = _[:, input_ids.shape[-1] :]

output = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

print(output)


A lei brasileira que mais se aprovou no ano de 1967 foi a Lei nº 5.254, de 4 de abril de 1967, com 108 parágrafos, sobre a preservação dos recifes de corais na Amazônia Azul. Essa lei criou o Conselho Nacional de Desenvolvimento Econômico (CONDENE), órgão responsável por formular e executar a política nacional de desenvolvimento econômico, e estabeleceu novas regras para a pesca na Amazônia Azul.

###


Reduziu o tempo de resposta da pergunta grande, talvez seja útil!

## Testando `say_less`

In [ ]:
# Copied threshold from /factscore_a=1_alpha=0.15_conf=frequency+gpt.txt.
# Compute new ones by running factscore.py with desired parameters and setting compute_single_threshold=True
threshold = 4.8998812119930735
merged_output, (accepted_subclaims, all_subclaims) = say_less(
    model, tokenizer, prompt, output, threshold
)
print("Original output: ")
print(output)
print("\n\n\n\n\nModified output: ")
print(merged_output)
print("\n\n\n\nAccepted sub-claims: ")
print(accepted_subclaims)
print("\n\n\n\nAll sub-claims: ")
print(all_subclaims)

Expecting value: line 1 column 5 (char 4)
Failed to parse as jsonl

    Você é um assistente prestativo cuja função é dividir suas entradas em um conjunto de pequenas afirmações, para que um ser humano possa verificar facilmente cada uma delas. Certifique-se de que cada afirmação seja pequena e não sobreposta às demais.

    Aqui está a entrada que você precisa dividir:
    Por favor, divida a seguinte entrada em um conjunto de pequenas afirmações independentes e retorne a saída no formato jsonl, onde cada linha seja {subclaim:[AFIRMAÇÃO], cocoruta-score:[CONF]}. A pontuação de confiança [CONF] deve representar o seu nível de confiança na afirmação, onde 1 corresponde a fatos e resultados óbvios, como 'A Terra é redonda' e '1+1=2'. Já 0 corresponde a afirmações muito obscuras ou difíceis de qualquer pessoa saber, como a data de aniversário de pessoas não públicas. A entrada é: 
Você é um assistente prestativo cuja função é dividir suas entradas em um conjunto de pequenas afirmações, pa

TypeError: object of type 'NoneType' has no len()

Dúvidas:
- Como usar device_map="auto" para usar as 4 GPUs? Tem alguma limitação prática?